# PSGC Bronze Ingestion

Ingests Philippine Standard Geographic Code (PSGC) reference data
from the PSA API into the Bronze layer.

## Design Goals

- Incremental/version-aware ingestion
- Idempotent reruns
- Raw source preservation
- Data-quality validation before Bronze promotion
- Batch-level lineage and auditability

In [0]:
import json
import hashlib
import uuid

import requests
import yaml

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from pyspark.sql import functions as F

In [0]:
# ============================================================
# CELL 2 — REUSABLE HELPER FUNCTIONS
# ============================================================


# ------------------------------------------------------------
# HTTP SESSION WITH RETRIES
# ------------------------------------------------------------

retry_strategy = Retry(
    total=3,
    connect=3,
    read=3,
    status=3,
    backoff_factor=2,
    status_forcelist=[
        429,
        500,
        502,
        503,
        504,
    ],
    allowed_methods=["GET"],
    raise_on_status=False,
)

session = requests.Session()

session.mount(
    "https://",
    HTTPAdapter(
        max_retries=retry_strategy
    ),
)


# ------------------------------------------------------------
# SQL-SAFE STRING HELPER
# ------------------------------------------------------------

def sql_quote(value):
    """
    Convert a Python value into a safely quoted SQL string
    for the controlled metadata SQL statements used
    in this notebook.
    """

    if value is None:
        return "NULL"

    escaped = str(value).replace(
        "'",
        "''",
    )

    return f"'{escaped}'"


# ------------------------------------------------------------
# FETCH ONE PSGC API PAGE
# ------------------------------------------------------------

def fetch_psgc_page(
    api_url,
    token,
    page,
    page_size,
):
    """
    Fetch one page from the PSA PSGC API.

    Returns:
        payload   - parsed JSON response
        raw_text  - original response body

    The API token is never printed or returned in logs.
    """

    params = {
        "token": token,
        "page": page,
        "page_size": page_size,
    }

    response = session.get(
        api_url,
        params=params,
        timeout=60,
    )

    status_code = response.status_code

    content_type = response.headers.get(
        "Content-Type",
        "",
    )


    # Do not print response.url because
    # the token may be present in the query string.
    if status_code != 200:
        raise RuntimeError(
            f"PSA API request failed "
            f"with HTTP {status_code}."
        )


    if "application/json" not in content_type.lower():
        raise RuntimeError(
            f"Expected JSON response but received "
            f"{content_type}."
        )


    try:
        payload = response.json()

    except ValueError as exc:
        raise RuntimeError(
            "PSA response could not be parsed as JSON."
        ) from exc


    return payload, response.text


# ------------------------------------------------------------
# EXTRACT PSGC RECORDS FROM API RESPONSE
# ------------------------------------------------------------

def extract_psgc_records(payload):
    """
    Extract PSGC records from the PSA paginated response.

    Actual response structure observed:

    {
        "count": ...,
        "next": ...,
        "previous": ...,
        "results": [...]
    }
    """

    if not isinstance(payload, dict):
        raise TypeError(
            "Unexpected PSA response type: "
            f"{type(payload)}"
        )


    results = payload.get(
        "results"
    )


    # Actual response structure observed
    if isinstance(results, list):
        return results


    # Defensive fallback in case another API version
    # returns a nested results object.
    if isinstance(results, dict):

        psgc_data = results.get(
            "psgc_data"
        )

        if isinstance(
            psgc_data,
            list,
        ):
            return psgc_data


        nested_data = results.get(
            "data"
        )

        if isinstance(
            nested_data,
            list,
        ):
            return nested_data


    raise ValueError(
        "Could not locate PSGC records. "
        f"Top-level keys: {list(payload.keys())}"
    )


# ------------------------------------------------------------
# UPDATE INGESTION METADATA
# ------------------------------------------------------------

def update_run(
    metadata_table,
    run_id,
    status,
    content_checksum=None,
    raw_path=None,
    record_count=None,
    page_count=None,
):
    """
    Update the metadata row for a specific ingestion run.

    Parameters:
        metadata_table:
            Fully qualified Delta metadata table name.

        run_id:
            Unique ingestion run identifier.

        status:
            Current run status such as:
            in_progress
            processed
            failed
            skipped_duplicate

        content_checksum:
            Logical SHA-256 checksum for the source snapshot.

        raw_path:
            Temporary or validated raw landing location.

        record_count:
            Number of source records processed.

        page_count:
            Number of API pages processed.
    """

    updates = [
        f"status = {sql_quote(status)}"
    ]


    if content_checksum is not None:
        updates.append(
            "content_checksum = "
            f"{sql_quote(content_checksum)}"
        )


    if raw_path is not None:
        updates.append(
            "raw_file_path = "
            f"{sql_quote(raw_path)}"
        )


    if record_count is not None:
        updates.append(
            "record_count = "
            f"{int(record_count)}"
        )


    if page_count is not None:
        updates.append(
            "page_count = "
            f"{int(page_count)}"
        )


    set_clause = ", ".join(
        updates
    )


    spark.sql(f"""
        UPDATE {metadata_table}
        SET {set_clause}
        WHERE batch_id = {sql_quote(run_id)}
    """)


# ------------------------------------------------------------
# NOTE
# ------------------------------------------------------------
#
# We intentionally do NOT place RUN_ID, METADATA_TABLE,
# SOURCE_NAME, or other run-specific variables inside these
# helpers.
#
# Those values are passed explicitly by later cells so that
# the helpers remain reusable and easier to debug.


print(
    "Cell 2 complete: "
    "reusable helper functions loaded successfully."
)

## 1. Configuration and Authentication

Loads non-sensitive source configuration from `sources.yaml`
and retrieves the PSA API token from Databricks Secrets.

In [0]:
DEFAULT_CONFIG_PATH = (
    "/Workspace/Users/"
    "shiena.arico@ftwfoundation.org/"
    "philippine-public-health-platform/"
    "config/sources.yaml"
)

try:
    CONFIG_PATH = dbutils.widgets.get(
        "config_path"
    )

except Exception:
    dbutils.widgets.text(
        "config_path",
        DEFAULT_CONFIG_PATH,
        "Config path",
    )

    CONFIG_PATH = dbutils.widgets.get(
        "config_path"
    )

CONFIG_PATH = CONFIG_PATH.strip()

if not CONFIG_PATH:
    raise ValueError(
        "config_path cannot be empty."
    )


with open(
    CONFIG_PATH,
    "r",
    encoding="utf-8",
) as file:
    config = yaml.safe_load(file)


psgc_cfg = config["sources"]["psgc"]

SOURCE_NAME = psgc_cfg["source_name"]
BASE_URL = psgc_cfg["base_url"]
SOURCE_VERSION = psgc_cfg["version"]
ENDPOINT = psgc_cfg["endpoint"]

PAGE_SIZE = int(
    psgc_cfg.get(
        "page_size",
        1000,
    )
)

LANDING_PATH = psgc_cfg[
    "landing_path"
]

BRONZE_TABLE = psgc_cfg[
    "bronze_table"
]

METADATA_TABLE = psgc_cfg[
    "metadata_table"
]

RAW_ROOT = (
    f"{LANDING_PATH.rstrip('/')}/raw"
)

print("Config loaded successfully")
print("Source:", SOURCE_NAME)
print("Base URL:", BASE_URL)
print("Version:", SOURCE_VERSION)
print("Endpoint:", ENDPOINT)
print("Page size:", PAGE_SIZE)
print("Landing path:", LANDING_PATH)
print("Raw root:", RAW_ROOT)
print("Bronze table:", BRONZE_TABLE)
print("Metadata table:", METADATA_TABLE)

In [0]:
PSA_TOKEN = dbutils.secrets.get(
    scope="g5-public-health",
    key="psa-psgc-token",
)

if not PSA_TOKEN:
    raise RuntimeError(
        "PSA API token could not be loaded."
    )

print(
    "PSA token loaded securely:",
    bool(PSA_TOKEN)
)

In [0]:
# ============================================================
# CELL 5 — BUILD API URL + INITIALIZE INGESTION RUN
# ============================================================

# ------------------------------------------------------------
# Build PSA API endpoint from configuration
# ------------------------------------------------------------

API_URL = (
    f"{BASE_URL.rstrip('/')}/"
    f"{SOURCE_VERSION}/"
    f"{ENDPOINT}"
)


# ------------------------------------------------------------
# Generate a unique identifier for this ingestion attempt
# ------------------------------------------------------------

RUN_ID = str(
    uuid.uuid4()
)


# ------------------------------------------------------------
# Temporary landing location
#
# Data is written here first while the batch is still being
# fetched and validated.
#
# Example:
# raw/_tmp/run_id=<uuid>
# ------------------------------------------------------------

TEMP_PATH = (
    f"{RAW_ROOT}/_tmp/"
    f"run_id={RUN_ID}"
)


# ------------------------------------------------------------
# Create temporary landing directory
# ------------------------------------------------------------

dbutils.fs.mkdirs(
    TEMP_PATH
)


# ------------------------------------------------------------
# Register the ingestion attempt
#
# Initial status = in_progress
#
# content_checksum is NULL because the complete logical
# dataset has not yet been retrieved and hashed.
# ------------------------------------------------------------

spark.sql(f"""
    INSERT INTO {METADATA_TABLE} (
        batch_id,
        source_name,
        source_version,
        endpoint,
        content_checksum,
        raw_file_path,
        record_count,
        page_count,
        retrieved_at,
        status
    )
    VALUES (
        {sql_quote(RUN_ID)},
        {sql_quote(SOURCE_NAME)},
        {sql_quote(SOURCE_VERSION)},
        {sql_quote(ENDPOINT)},
        NULL,
        {sql_quote(TEMP_PATH)},
        0,
        0,
        current_timestamp(),
        'in_progress'
    )
""")


# ------------------------------------------------------------
# Confirmation
# ------------------------------------------------------------

print("-" * 60)
print("CELL 5 COMPLETE — INGESTION RUN INITIALIZED")
print("-" * 60)

print(
    "API URL:",
    API_URL
)

print(
    "Run ID:",
    RUN_ID
)

print(
    "Temporary landing path:",
    TEMP_PATH
)

print(
    "Metadata table:",
    METADATA_TABLE
)

print(
    "Status: in_progress"
)

In [0]:
# ============================================================
# CELL 6 — TEST API + VALIDATE RESPONSE STRUCTURE
# ============================================================

try:

    # --------------------------------------------------------
    # Request the first page only.
    #
    # Purpose:
    # Verify authentication, endpoint availability,
    # JSON response structure, and source fields before
    # downloading the complete dataset.
    # --------------------------------------------------------

    test_payload, _ = fetch_psgc_page(
        api_url=API_URL,
        token=PSA_TOKEN,
        page=1,
        page_size=PAGE_SIZE,
    )


    # --------------------------------------------------------
    # Validate top-level response structure
    # --------------------------------------------------------

    if not isinstance(test_payload, dict):
        raise TypeError(
            "Expected PSA API response to be a dictionary."
        )


    required_response_fields = {
        "count",
        "next",
        "previous",
        "results",
    }


    missing_response_fields = (
        required_response_fields
        - set(test_payload.keys())
    )


    if missing_response_fields:
        raise ValueError(
            "PSA response is missing expected fields: "
            f"{sorted(missing_response_fields)}"
        )


    # --------------------------------------------------------
    # Extract page records using reusable helper
    # --------------------------------------------------------

    test_records = extract_psgc_records(
        test_payload
    )


    if not test_records:
        raise ValueError(
            "PSA API returned zero records on the test page."
        )


    # --------------------------------------------------------
    # Validate expected source fields
    #
    # These are based on the actual PSA response we observed.
    # --------------------------------------------------------

    first_record = test_records[0]

    required_record_fields = {
        "code",
        "area_name",
        "geographic_level",
        "version",
    }


    missing_record_fields = (
        required_record_fields
        - set(first_record.keys())
    )


    if missing_record_fields:
        raise ValueError(
            "PSA record is missing expected fields: "
            f"{sorted(missing_record_fields)}"
        )


    # --------------------------------------------------------
    # Successful API test
    # --------------------------------------------------------

    print("-" * 60)
    print("CELL 6 COMPLETE — PSA API VALIDATED")
    print("-" * 60)

    print(
        "Total records reported by API:",
        test_payload.get("count")
    )

    print(
        "Records on first page:",
        len(test_records)
    )

    print(
        "Has next page:",
        test_payload.get("next") is not None
    )

    print(
        "First record fields:",
        list(first_record.keys())
    )


except Exception:

    # --------------------------------------------------------
    # Mark this ingestion attempt as failed
    # --------------------------------------------------------

    update_run(
        metadata_table=METADATA_TABLE,
        run_id=RUN_ID,
        status="failed",
    )

    raise

## 2. Extraction

Retrieves PSGC records using paginated API requests.

Each API page is persisted independently to the landing Volume
to avoid accumulating the complete source payload in driver memory.

In [0]:
# ============================================================
# CELL 7 — PAGINATED EXTRACTION + TEMPORARY RAW LANDING
# ============================================================

PAGE_COUNT = 0
RECORD_COUNT = 0
EXPECTED_TOTAL_COUNT = None

MAX_PAGES = 10000


# ------------------------------------------------------------
# Small in-memory state used for validation and checksum.
#
# We do NOT store all source records in one Python list.
# ------------------------------------------------------------

record_hashes = []

seen_codes = set()
duplicate_codes = set()
seen_page_signatures = set()

missing_code_count = 0
version_mismatch_count = 0


try:

    page = 1


    while page <= MAX_PAGES:

        # ----------------------------------------------------
        # Fetch one API page
        # ----------------------------------------------------

        payload, raw_text = fetch_psgc_page(
            api_url=API_URL,
            token=PSA_TOKEN,
            page=page,
            page_size=PAGE_SIZE,
        )


        records = extract_psgc_records(
            payload
        )


        # ----------------------------------------------------
        # Capture API-reported total record count
        # ----------------------------------------------------

        if EXPECTED_TOTAL_COUNT is None:
            EXPECTED_TOTAL_COUNT = payload.get(
                "count"
            )


        print(
            f"Page {page}: "
            f"{len(records)} records"
        )


        # ----------------------------------------------------
        # Defensive stop if API returns an empty page
        # ----------------------------------------------------

        if len(records) == 0:
            break


        # ----------------------------------------------------
        # Detect accidental repeated API pages
        #
        # Source natural key = code
        # ----------------------------------------------------

        page_codes = [
            str(
                record.get(
                    "code",
                    ""
                )
            )
            for record in records
        ]


        page_signature = hashlib.sha256(
            json.dumps(
                page_codes,
                separators=(",", ":"),
                ensure_ascii=False,
            ).encode("utf-8")
        ).hexdigest()


        if page_signature in seen_page_signatures:
            raise RuntimeError(
                f"Repeated API page detected "
                f"at page {page}."
            )


        seen_page_signatures.add(
            page_signature
        )


        # ----------------------------------------------------
        # Save exact API response for audit / lineage
        #
        # Example:
        # page_0001.json
        # ----------------------------------------------------

        raw_page_file = (
            f"{TEMP_PATH}/"
            f"page_{page:04d}.json"
        )


        with open(
            raw_page_file,
            "w",
            encoding="utf-8",
        ) as raw_output:

            raw_output.write(
                raw_text
            )


        # ----------------------------------------------------
        # Create one NDJSON file for this API page
        #
        # Example:
        # records_page_0001.ndjson
        #
        # One logical PSGC record per line.
        # ----------------------------------------------------

        records_page_file = (
            f"{TEMP_PATH}/"
            f"records_page_{page:04d}.ndjson"
        )


        with open(
            records_page_file,
            "w",
            encoding="utf-8",
        ) as records_output:


            for record in records:

                # --------------------------------------------
                # Validate source record structure
                # --------------------------------------------

                if not isinstance(
                    record,
                    dict,
                ):
                    raise ValueError(
                        f"Page {page} contains "
                        "a record that is not "
                        "a JSON object."
                    )


                # --------------------------------------------
                # Create deterministic representation
                # --------------------------------------------

                canonical_record = json.dumps(
                    record,
                    sort_keys=True,
                    separators=(",", ":"),
                    ensure_ascii=False,
                )


                # --------------------------------------------
                # Persist logical source record
                # --------------------------------------------

                records_output.write(
                    canonical_record
                    + "\n"
                )


                # --------------------------------------------
                # Generate record-level hash
                #
                # These hashes will later be used to create
                # the logical dataset checksum.
                # --------------------------------------------

                record_hash = hashlib.sha256(
                    canonical_record.encode(
                        "utf-8"
                    )
                ).hexdigest()


                record_hashes.append(
                    record_hash
                )


                # --------------------------------------------
                # Validate required source key
                #
                # Actual PSA field = code
                # --------------------------------------------

                source_code = record.get(
                    "code"
                )


                if not source_code:

                    missing_code_count += 1


                else:

                    source_code = str(
                        source_code
                    )


                    if source_code in seen_codes:

                        duplicate_codes.add(
                            source_code
                        )


                    seen_codes.add(
                        source_code
                    )


                # --------------------------------------------
                # Validate source version
                # --------------------------------------------

                record_version = record.get(
                    "version"
                )


                if (
                    record_version is not None
                    and
                    str(record_version)
                    != SOURCE_VERSION
                ):

                    version_mismatch_count += 1


                RECORD_COUNT += 1


        # ----------------------------------------------------
        # Current page successfully persisted
        # ----------------------------------------------------

        PAGE_COUNT += 1


        # ----------------------------------------------------
        # PSA pagination metadata
        #
        # next = None means current page is the final page.
        # ----------------------------------------------------

        if payload.get("next") is None:
            break


        page += 1


    # --------------------------------------------------------
    # Pagination safety limit
    # --------------------------------------------------------

    if page > MAX_PAGES:

        raise RuntimeError(
            f"Pagination exceeded the safety limit "
            f"of {MAX_PAGES} pages."
        )


    # --------------------------------------------------------
    # Extraction summary
    # --------------------------------------------------------

    print("-" * 60)
    print(
        "CELL 7 COMPLETE — EXTRACTION FINISHED"
    )
    print("-" * 60)

    print(
        "API expected total:",
        EXPECTED_TOTAL_COUNT
    )

    print(
        "Pages retrieved:",
        PAGE_COUNT
    )

    print(
        "Records retrieved:",
        RECORD_COUNT
    )

    print(
        "Unique source codes:",
        len(seen_codes)
    )


except Exception:

    update_run(
        metadata_table=METADATA_TABLE,
        run_id=RUN_ID,
        status="failed",
        raw_path=TEMP_PATH,
        record_count=RECORD_COUNT,
        page_count=PAGE_COUNT,
    )

    raise

## 3. Landing Quality Gate

Validates:

- source-reported row count
- missing geographic codes
- duplicate geographic codes
- source version consistency
- number of raw API pages
- number of landed NDJSON pages

In [0]:
# ============================================================
# CELL 8 — LANDING DATA QUALITY VALIDATION
# ============================================================

try:

    # --------------------------------------------------------
    # CHECK 1 — Source must contain records
    # --------------------------------------------------------

    if RECORD_COUNT == 0:
        raise ValueError(
            "PSGC API returned zero records."
        )


    # --------------------------------------------------------
    # CHECK 2 — API total vs retrieved total
    # --------------------------------------------------------

    print(
        "API expected total:",
        EXPECTED_TOTAL_COUNT
    )

    print(
        "Retrieved total:",
        RECORD_COUNT
    )


    if (
        EXPECTED_TOTAL_COUNT is not None
        and RECORD_COUNT != EXPECTED_TOTAL_COUNT
    ):
        raise RuntimeError(
            "Source row-count reconciliation failed. "
            f"API expected={EXPECTED_TOTAL_COUNT}, "
            f"retrieved={RECORD_COUNT}"
        )


    # --------------------------------------------------------
    # CHECK 3 — Missing source codes
    # --------------------------------------------------------

    print(
        "Missing source codes:",
        missing_code_count
    )


    if missing_code_count > 0:
        raise ValueError(
            f"Found {missing_code_count} "
            "records without code."
        )


    # --------------------------------------------------------
    # CHECK 4 — Duplicate source codes
    # --------------------------------------------------------

    print(
        "Duplicate source codes:",
        len(duplicate_codes)
    )


    if duplicate_codes:

        sample_duplicates = list(
            duplicate_codes
        )[:10]

        raise ValueError(
            f"Found {len(duplicate_codes)} "
            "duplicate source codes. "
            f"Examples: {sample_duplicates}"
        )


    # --------------------------------------------------------
    # CHECK 5 — Source version consistency
    # --------------------------------------------------------

    print(
        "Version mismatches:",
        version_mismatch_count
    )


    if version_mismatch_count > 0:
        raise ValueError(
            f"Found {version_mismatch_count} "
            "records whose version does not match "
            f"{SOURCE_VERSION}."
        )


    # --------------------------------------------------------
    # CHECK 6 — Validate landed NDJSON record count
    # --------------------------------------------------------

    LANDED_RECORD_COUNT = (
        spark.read
        .text(
            f"{TEMP_PATH}/records_page_*.ndjson"
        )
        .count()
    )


    print(
        "Retrieved records:",
        RECORD_COUNT
    )

    print(
        "Landed NDJSON records:",
        LANDED_RECORD_COUNT
    )


    if LANDED_RECORD_COUNT != RECORD_COUNT:
        raise RuntimeError(
            "Landing row-count reconciliation failed. "
            f"Retrieved={RECORD_COUNT}, "
            f"landed={LANDED_RECORD_COUNT}"
        )


    # --------------------------------------------------------
    # CHECK 7 — Count raw API page files
    # --------------------------------------------------------

    temp_files = dbutils.fs.ls(
        TEMP_PATH
    )


    raw_page_files = [
        item
        for item in temp_files
        if (
            item.name.startswith("page_")
            and item.name.endswith(".json")
        )
    ]


    RAW_PAGE_COUNT = len(
        raw_page_files
    )


    print(
        "Expected page count:",
        PAGE_COUNT
    )

    print(
        "Raw API page files:",
        RAW_PAGE_COUNT
    )


    if RAW_PAGE_COUNT != PAGE_COUNT:
        raise RuntimeError(
            "Raw page reconciliation failed. "
            f"Expected={PAGE_COUNT}, "
            f"actual={RAW_PAGE_COUNT}"
        )


    # --------------------------------------------------------
    # CHECK 8 — Count NDJSON page files
    # --------------------------------------------------------

    ndjson_page_files = [
        item
        for item in temp_files
        if (
            item.name.startswith(
                "records_page_"
            )
            and item.name.endswith(
                ".ndjson"
            )
        )
    ]


    NDJSON_PAGE_COUNT = len(
        ndjson_page_files
    )


    print(
        "NDJSON page files:",
        NDJSON_PAGE_COUNT
    )


    if NDJSON_PAGE_COUNT != PAGE_COUNT:
        raise RuntimeError(
            "NDJSON page reconciliation failed. "
            f"Expected={PAGE_COUNT}, "
            f"actual={NDJSON_PAGE_COUNT}"
        )


    # --------------------------------------------------------
    # CHECK 9 — Unique code reconciliation
    # --------------------------------------------------------

    UNIQUE_CODE_COUNT = len(
        seen_codes
    )


    print(
        "Unique source codes:",
        UNIQUE_CODE_COUNT
    )


    if UNIQUE_CODE_COUNT != RECORD_COUNT:
        raise RuntimeError(
            "Source code uniqueness reconciliation failed. "
            f"Records={RECORD_COUNT}, "
            f"unique codes={UNIQUE_CODE_COUNT}"
        )


    # --------------------------------------------------------
    # Mark extraction + landing validation complete
    # --------------------------------------------------------

    dbutils.fs.put(
        f"{TEMP_PATH}/_FETCH_COMPLETE",
        (
            f"records={RECORD_COUNT}\n"
            f"pages={PAGE_COUNT}\n"
            f"source_version={SOURCE_VERSION}\n"
        ),
        overwrite=True,
    )


    # --------------------------------------------------------
    # Keep ingestion run in progress
    # --------------------------------------------------------

    update_run(
        metadata_table=METADATA_TABLE,
        run_id=RUN_ID,
        status="in_progress",
        raw_path=TEMP_PATH,
        record_count=RECORD_COUNT,
        page_count=PAGE_COUNT,
    )


    print("-" * 60)
    print(
        "CELL 8 COMPLETE — LANDING VALIDATION PASSED"
    )
    print("-" * 60)

    print(
        "Validated records:",
        RECORD_COUNT
    )

    print(
        "Validated pages:",
        PAGE_COUNT
    )

    print(
        "Unique codes:",
        UNIQUE_CODE_COUNT
    )


except Exception:

    update_run(
        metadata_table=METADATA_TABLE,
        run_id=RUN_ID,
        status="failed",
        raw_path=TEMP_PATH,
        record_count=RECORD_COUNT,
        page_count=PAGE_COUNT,
    )

    raise

## 4. Idempotency

A deterministic SHA-256 checksum is computed from the logical
source records.

Previously processed content with the same checksum is skipped,
preventing duplicate Bronze ingestion.

In [0]:
# ============================================================
# CELL 9 — LOGICAL CONTENT CHECKSUM
# ============================================================

try:

    # --------------------------------------------------------
    # Safety checks
    # --------------------------------------------------------

    if not record_hashes:
        raise RuntimeError(
            "No record hashes are available "
            "for checksum calculation."
        )


    if len(record_hashes) != RECORD_COUNT:
        raise RuntimeError(
            "Record hash reconciliation failed. "
            f"Records={RECORD_COUNT}, "
            f"hashes={len(record_hashes)}"
        )


    # --------------------------------------------------------
    # Sort hashes so source/page ordering does not affect
    # the logical dataset checksum
    # --------------------------------------------------------

    sorted_hashes = sorted(
        record_hashes
    )


    # --------------------------------------------------------
    # Build deterministic checksum input
    # --------------------------------------------------------

    checksum_input = "\n".join(
        sorted_hashes
    )


    # --------------------------------------------------------
    # Create dataset-level SHA-256 checksum
    # --------------------------------------------------------

    CONTENT_CHECKSUM = hashlib.sha256(
        checksum_input.encode(
            "utf-8"
        )
    ).hexdigest()


    # --------------------------------------------------------
    # Deterministic final raw location
    #
    # Same logical dataset =
    # same checksum =
    # same final raw path
    # --------------------------------------------------------

    FINAL_RAW_PATH = (
        f"{RAW_ROOT}/"
        f"version={SOURCE_VERSION}/"
        f"content_checksum={CONTENT_CHECKSUM}"
    )


    # --------------------------------------------------------
    # Persist checksum in ingestion metadata
    #
    # Status remains in_progress because Bronze has
    # not completed yet.
    # --------------------------------------------------------

    update_run(
        metadata_table=METADATA_TABLE,
        run_id=RUN_ID,
        status="in_progress",
        content_checksum=CONTENT_CHECKSUM,
        raw_path=TEMP_PATH,
        record_count=RECORD_COUNT,
        page_count=PAGE_COUNT,
    )


    print("-" * 60)
    print(
        "CELL 9 COMPLETE — CONTENT CHECKSUM CREATED"
    )
    print("-" * 60)

    print(
        "Record hashes:",
        len(record_hashes)
    )

    print(
        "Content checksum:",
        CONTENT_CHECKSUM
    )

    print(
        "Final raw path:",
        FINAL_RAW_PATH
    )


except Exception:

    update_run(
        metadata_table=METADATA_TABLE,
        run_id=RUN_ID,
        status="failed",
        raw_path=TEMP_PATH,
        record_count=RECORD_COUNT,
        page_count=PAGE_COUNT,
    )

    raise

In [0]:
# ============================================================
# CELL 10 — IDEMPOTENCY CHECK
# ============================================================

try:

    # --------------------------------------------------------
    # Look for a PREVIOUS successful run with the same:
    #
    # - source
    # - source version
    # - endpoint
    # - logical content checksum
    #
    # Exclude the current RUN_ID.
    # --------------------------------------------------------

    existing_batch = (
        spark.table(METADATA_TABLE)
        .filter(
            (F.col("batch_id") != RUN_ID)
            &
            (F.col("source_name") == SOURCE_NAME)
            &
            (F.col("source_version") == SOURCE_VERSION)
            &
            (F.col("endpoint") == ENDPOINT)
            &
            (F.col("content_checksum") == CONTENT_CHECKSUM)
            &
            (F.col("status") == "processed")
        )
        .select(
            "batch_id",
            "raw_file_path",
            "record_count",
            "page_count",
        )
        .limit(1)
        .collect()
    )

except Exception:

    update_run(
        metadata_table=METADATA_TABLE,
        run_id=RUN_ID,
        status="failed",
        content_checksum=CONTENT_CHECKSUM,
        raw_path=TEMP_PATH,
        record_count=RECORD_COUNT,
        page_count=PAGE_COUNT,
    )

    raise


# ------------------------------------------------------------
# DUPLICATE CONTENT FOUND
# ------------------------------------------------------------

if existing_batch:

    previous_run = existing_batch[0]

    previous_batch_id = (
        previous_run["batch_id"]
    )

    previous_raw_path = (
        previous_run["raw_file_path"]
    )


    print("-" * 60)
    print("DUPLICATE CONTENT DETECTED")
    print("-" * 60)

    print(
        "Current run:",
        RUN_ID
    )

    print(
        "Previously processed run:",
        previous_batch_id
    )

    print(
        "Content checksum:",
        CONTENT_CHECKSUM
    )

    print(
        "Records:",
        RECORD_COUNT
    )

    print(
        "Pages:",
        PAGE_COUNT
    )


    # --------------------------------------------------------
    # Remove temporary files created by this duplicate run
    #
    # Permanent raw data already exists from the successful
    # previous run.
    # --------------------------------------------------------

    try:

        dbutils.fs.rm(
            TEMP_PATH,
            recurse=True,
        )

        print(
            "Temporary duplicate files removed."
        )

    except Exception as cleanup_error:

        # Cleanup failure should not change our conclusion
        # that the logical dataset is already processed.
        print(
            "Warning: temporary cleanup failed:"
        )

        print(
            str(cleanup_error)
        )


    # --------------------------------------------------------
    # Preserve this ingestion attempt for auditability
    # --------------------------------------------------------

    update_run(
        metadata_table=METADATA_TABLE,
        run_id=RUN_ID,
        status="skipped_duplicate",
        content_checksum=CONTENT_CHECKSUM,
        raw_path=previous_raw_path,
        record_count=RECORD_COUNT,
        page_count=PAGE_COUNT,
    )


    print("-" * 60)
    print(
        "CELL 10 COMPLETE — INGESTION SKIPPED"
    )
    print("-" * 60)

    print(
        "Reason: identical PSGC content "
        "was already processed."
    )

    print(
        "Status: skipped_duplicate"
    )


    # --------------------------------------------------------
    # Stop notebook intentionally.
    #
    # Cells 11–13 must NOT run for duplicate content.
    # --------------------------------------------------------

    dbutils.notebook.exit(
        "SKIPPED_DUPLICATE: "
        "Same PSGC content was already processed."
    )


# ------------------------------------------------------------
# NEW CONTENT
# ------------------------------------------------------------

print("-" * 60)
print(
    "CELL 10 COMPLETE — NEW CONTENT DETECTED"
)
print("-" * 60)

print(
    "No previously processed batch "
    "has this content checksum."
)

print(
    "Continue to Cell 11."
)

In [0]:
# ============================================================
# CELL 11 — PROMOTE VALIDATED RAW BATCH
# ============================================================

def dbfs_path_exists(path):
    """
    Return True if a Databricks/Volume path exists.
    """
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


try:

    # --------------------------------------------------------
    # Parent directory for this PSGC version
    # --------------------------------------------------------

    VERSION_RAW_PATH = (
        f"{RAW_ROOT}/"
        f"version={SOURCE_VERSION}"
    )

    dbutils.fs.mkdirs(
        VERSION_RAW_PATH
    )


    # --------------------------------------------------------
    # Check whether the deterministic final path already exists
    # --------------------------------------------------------

    final_path_exists = dbfs_path_exists(
        FINAL_RAW_PATH
    )

    success_marker_path = (
        f"{FINAL_RAW_PATH}/_SUCCESS"
    )

    success_marker_exists = dbfs_path_exists(
        success_marker_path
    )


    # --------------------------------------------------------
    # CASE 1:
    # A previously validated raw copy already exists.
    #
    # Example:
    # raw promotion succeeded in an earlier run,
    # but Bronze failed afterward.
    #
    # Reuse the existing raw files instead of copying them again.
    # --------------------------------------------------------

    if (
        final_path_exists
        and success_marker_exists
    ):

        print(
            "Validated raw batch already exists."
        )

        print(
            "Reusing:",
            FINAL_RAW_PATH
        )

        # Current run's temporary copy is no longer needed.
        if dbfs_path_exists(TEMP_PATH):

            dbutils.fs.rm(
                TEMP_PATH,
                recurse=True,
            )

            print(
                "Temporary duplicate raw files removed."
            )


    # --------------------------------------------------------
    # CASE 2:
    # Final path exists but does NOT have _SUCCESS.
    #
    # Treat it as an incomplete previous promotion.
    # --------------------------------------------------------

    elif (
        final_path_exists
        and not success_marker_exists
    ):

        print(
            "Incomplete final raw path detected."
        )

        print(
            "Removing incomplete path:",
            FINAL_RAW_PATH
        )

        dbutils.fs.rm(
            FINAL_RAW_PATH,
            recurse=True,
        )


        # Move the current validated temporary batch
        # into the deterministic final raw location.
        dbutils.fs.mv(
            TEMP_PATH,
            FINAL_RAW_PATH,
            recurse=True,
        )

        print(
            "Validated temporary batch promoted."
        )


    # --------------------------------------------------------
    # CASE 3:
    # Final location does not exist yet.
    # Normal first-run behavior.
    # --------------------------------------------------------

    else:

        dbutils.fs.mv(
            TEMP_PATH,
            FINAL_RAW_PATH,
            recurse=True,
        )

        print(
            "Validated temporary batch promoted."
        )


    # --------------------------------------------------------
    # Verify expected landed files exist after promotion
    # --------------------------------------------------------

    final_files = dbutils.fs.ls(
        FINAL_RAW_PATH
    )


    raw_json_files = [
        item
        for item in final_files
        if (
            item.name.startswith("page_")
            and item.name.endswith(".json")
        )
    ]


    ndjson_files = [
        item
        for item in final_files
        if (
            item.name.startswith("records_page_")
            and item.name.endswith(".ndjson")
        )
    ]


    print(
        "Final raw API page files:",
        len(raw_json_files)
    )

    print(
        "Final NDJSON page files:",
        len(ndjson_files)
    )


    # --------------------------------------------------------
    # Reconciliation before declaring raw promotion successful
    # --------------------------------------------------------

    if len(raw_json_files) != PAGE_COUNT:
        raise RuntimeError(
            "Final raw JSON page count mismatch. "
            f"Expected={PAGE_COUNT}, "
            f"actual={len(raw_json_files)}"
        )


    if len(ndjson_files) != PAGE_COUNT:
        raise RuntimeError(
            "Final NDJSON page count mismatch. "
            f"Expected={PAGE_COUNT}, "
            f"actual={len(ndjson_files)}"
        )


    # --------------------------------------------------------
    # Create permanent success marker
    #
    # This means:
    # extraction + landing + validation + raw promotion passed.
    #
    # It does NOT mean Bronze has completed yet.
    # --------------------------------------------------------

    dbutils.fs.put(
        success_marker_path,
        (
            f"source={SOURCE_NAME}\n"
            f"source_version={SOURCE_VERSION}\n"
            f"record_count={RECORD_COUNT}\n"
            f"page_count={PAGE_COUNT}\n"
            f"content_checksum={CONTENT_CHECKSUM}\n"
        ),
        overwrite=True,
    )


    # --------------------------------------------------------
    # Update ingestion state
    #
    # Still in_progress because Bronze is next.
    # --------------------------------------------------------

    update_run(
        status="in_progress",
        content_checksum=CONTENT_CHECKSUM,
        raw_path=FINAL_RAW_PATH,
        record_count=RECORD_COUNT,
        page_count=PAGE_COUNT,
    )


    print("-" * 60)
    print(
        "CELL 11 COMPLETE — RAW BATCH PROMOTED"
    )
    print("-" * 60)

    print(
        "Final raw path:",
        FINAL_RAW_PATH
    )

    print(
        "Raw API pages:",
        len(raw_json_files)
    )

    print(
        "NDJSON pages:",
        len(ndjson_files)
    )

    print(
        "Status: in_progress"
    )


except Exception:

    # At this point TEMP_PATH might already have been moved,
    # so preserve whichever path currently exists.
    failure_path = (
        FINAL_RAW_PATH
        if dbfs_path_exists(FINAL_RAW_PATH)
        else TEMP_PATH
    )

    update_run(
        status="failed",
        content_checksum=CONTENT_CHECKSUM,
        raw_path=failure_path,
        record_count=RECORD_COUNT,
        page_count=PAGE_COUNT,
    )

    raise

## 5. Bronze Load

Validated landed records are merged into the Bronze Delta table.

MERGE key:

`content_checksum + psgc_code`

This protects the pipeline from duplicate writes during retries.

In [0]:
# ============================================================
# CELL 12 — BRONZE MERGE
# ============================================================

try:

    # --------------------------------------------------------
    # Read all validated NDJSON files from the promoted
    # checksum-based raw location.
    # --------------------------------------------------------

    RECORDS_PATH = (
        f"{FINAL_RAW_PATH}/"
        "records_page_*.ndjson"
    )

    records_df = (
        spark.read
        .text(RECORDS_PATH)
    )


    # --------------------------------------------------------
    # Parse PSA source fields.
    #
    # Bronze stays source-aligned:
    # - useful source attributes are exposed as columns
    # - populations remains JSON because it contains
    #   multiple population observations per PSGC code
    # - full source record is also retained
    # --------------------------------------------------------

    bronze_df = (
        records_df

        .select(

            # =================================================
            # PSA SOURCE FIELDS
            # =================================================

            # PSA source field "code"
            # standardized to "psgc_code"
            F.get_json_object(
                F.col("value"),
                "$.code"
            ).alias(
                "psgc_code"
            ),

            F.get_json_object(
                F.col("value"),
                "$.area_name"
            ).alias(
                "area_name"
            ),

            F.get_json_object(
                F.col("value"),
                "$.correspondence_code"
            ).alias(
                "correspondence_code"
            ),

            F.get_json_object(
                F.col("value"),
                "$.geographic_level"
            ).alias(
                "geographic_level"
            ),


            # -------------------------------------------------
            # Geographic hierarchy
            # -------------------------------------------------

            F.get_json_object(
                F.col("value"),
                "$.reg"
            ).alias(
                "reg"
            ),

            F.get_json_object(
                F.col("value"),
                "$.prv"
            ).alias(
                "prv"
            ),

            F.get_json_object(
                F.col("value"),
                "$.mun"
            ).alias(
                "mun"
            ),

            F.get_json_object(
                F.col("value"),
                "$.bgy"
            ).alias(
                "bgy"
            ),


            # -------------------------------------------------
            # Additional PSA attributes
            # -------------------------------------------------

            F.get_json_object(
                F.col("value"),
                "$.old_name"
            ).alias(
                "old_name"
            ),

            F.get_json_object(
                F.col("value"),
                "$.city_class"
            ).alias(
                "city_class"
            ),

            F.get_json_object(
                F.col("value"),
                "$.income_classification"
            ).alias(
                "income_classification"
            ),

            F.get_json_object(
                F.col("value"),
                "$.urban_rural"
            ).alias(
                "urban_rural"
            ),

            F.get_json_object(
                F.col("value"),
                "$.island_region"
            ).alias(
                "island_region"
            ),

            # Rename PSA "status" so it is not confused
            # with ingestion run status.
            F.get_json_object(
                F.col("value"),
                "$.status"
            ).alias(
                "record_status"
            ),

            # Rename PSA record "version" so it is distinct
            # from pipeline/config source_version.
            F.get_json_object(
                F.col("value"),
                "$.version"
            ).alias(
                "record_version"
            ),


            # -------------------------------------------------
            # Nested population observations
            #
            # Example:
            # [
            #   {"year": 2015, ...},
            #   {"year": 2020, ...},
            #   {"year": 2024, ...}
            # ]
            #
            # Keep as JSON in Bronze.
            # Silver will parse/explode this later.
            # -------------------------------------------------

            F.get_json_object(
                F.col("value"),
                "$.populations"
            ).alias(
                "populations_json"
            ),


            # -------------------------------------------------
            # Complete logical source record
            # -------------------------------------------------

            F.col(
                "value"
            ).alias(
                "source_record_json"
            )
        )


        # =====================================================
        # TECHNICAL INGESTION METADATA
        # =====================================================

        .withColumn(
            "source_system",
            F.lit(SOURCE_NAME)
        )

        .withColumn(
            "source_version",
            F.lit(SOURCE_VERSION)
        )

        .withColumn(
            "source_endpoint",
            F.lit(ENDPOINT)
        )

        .withColumn(
            "batch_id",
            F.lit(RUN_ID)
        )

        .withColumn(
            "content_checksum",
            F.lit(CONTENT_CHECKSUM)
        )

        .withColumn(
            "raw_path",
            F.lit(FINAL_RAW_PATH)
        )

        .withColumn(
            "ingested_at",
            F.current_timestamp()
        )
    )


    # --------------------------------------------------------
    # CHECK 1 — Source vs Bronze incoming row count
    # --------------------------------------------------------

    BRONZE_INCOMING_COUNT = (
        bronze_df.count()
    )

    print(
        "Source records:",
        RECORD_COUNT
    )

    print(
        "Bronze incoming records:",
        BRONZE_INCOMING_COUNT
    )


    if BRONZE_INCOMING_COUNT != RECORD_COUNT:

        raise RuntimeError(
            "Bronze input reconciliation failed. "
            f"Expected={RECORD_COUNT}, "
            f"actual={BRONZE_INCOMING_COUNT}"
        )


    # --------------------------------------------------------
    # CHECK 2 — Required PSGC key must exist
    # --------------------------------------------------------

    BRONZE_MISSING_CODES = (
        bronze_df
        .filter(
            F.col("psgc_code").isNull()
            |
            (
                F.trim(
                    F.col("psgc_code")
                )
                == ""
            )
        )
        .count()
    )


    print(
        "Bronze missing PSGC codes:",
        BRONZE_MISSING_CODES
    )


    if BRONZE_MISSING_CODES > 0:

        raise RuntimeError(
            f"Bronze parsing produced "
            f"{BRONZE_MISSING_CODES} "
            "missing PSGC codes."
        )


    # --------------------------------------------------------
    # CHECK 3 — Incoming PSGC codes must be unique
    # within this logical source snapshot
    # --------------------------------------------------------

    BRONZE_INCOMING_UNIQUE_CODES = (
        bronze_df
        .select(
            "psgc_code"
        )
        .distinct()
        .count()
    )


    print(
        "Bronze incoming unique codes:",
        BRONZE_INCOMING_UNIQUE_CODES
    )


    if (
        BRONZE_INCOMING_UNIQUE_CODES
        != BRONZE_INCOMING_COUNT
    ):

        raise RuntimeError(
            "Bronze incoming uniqueness check failed. "
            f"Rows={BRONZE_INCOMING_COUNT}, "
            f"unique codes="
            f"{BRONZE_INCOMING_UNIQUE_CODES}"
        )


    # --------------------------------------------------------
    # Create temporary view for Delta MERGE
    # --------------------------------------------------------

    bronze_df.createOrReplaceTempView(
        "psgc_incoming"
    )


    # --------------------------------------------------------
    # Idempotent Bronze MERGE
    #
    # Grain:
    # one PSGC geographic code
    # per logical source snapshot
    #
    # Merge identity:
    # content_checksum + psgc_code
    # --------------------------------------------------------

    spark.sql(f"""
        MERGE INTO {BRONZE_TABLE} AS target

        USING psgc_incoming AS source

        ON
            target.content_checksum
                = source.content_checksum

            AND

            target.psgc_code
                = source.psgc_code


        WHEN NOT MATCHED THEN

            INSERT (

                -- PSA source fields
                psgc_code,
                area_name,
                correspondence_code,
                geographic_level,
                reg,
                prv,
                mun,
                bgy,
                old_name,
                city_class,
                income_classification,
                urban_rural,
                island_region,
                record_status,
                record_version,
                populations_json,

                -- Complete source record
                source_record_json,

                -- Technical metadata
                source_system,
                source_version,
                source_endpoint,
                batch_id,
                content_checksum,
                raw_path,
                ingested_at
            )

            VALUES (

                source.psgc_code,
                source.area_name,
                source.correspondence_code,
                source.geographic_level,
                source.reg,
                source.prv,
                source.mun,
                source.bgy,
                source.old_name,
                source.city_class,
                source.income_classification,
                source.urban_rural,
                source.island_region,
                source.record_status,
                source.record_version,
                source.populations_json,

                source.source_record_json,

                source.source_system,
                source.source_version,
                source.source_endpoint,
                source.batch_id,
                source.content_checksum,
                source.raw_path,
                source.ingested_at
            )
    """)


    # --------------------------------------------------------
    # Validate the exact logical snapshot after MERGE
    # --------------------------------------------------------

    bronze_snapshot_df = (
        spark.table(
            BRONZE_TABLE
        )
        .filter(
            F.col("content_checksum")
            == CONTENT_CHECKSUM
        )
    )


    BRONZE_BATCH_COUNT = (
        bronze_snapshot_df.count()
    )


    BRONZE_UNIQUE_CODES = (
        bronze_snapshot_df
        .select(
            "psgc_code"
        )
        .distinct()
        .count()
    )


    print("-" * 60)

    print(
        "Expected source records:",
        RECORD_COUNT
    )

    print(
        "Bronze stored records:",
        BRONZE_BATCH_COUNT
    )

    print(
        "Bronze unique PSGC codes:",
        BRONZE_UNIQUE_CODES
    )


    # --------------------------------------------------------
    # Final Bronze reconciliation
    # --------------------------------------------------------

    if BRONZE_BATCH_COUNT != RECORD_COUNT:

        raise RuntimeError(
            "Bronze row-count reconciliation failed. "
            f"Expected={RECORD_COUNT}, "
            f"actual={BRONZE_BATCH_COUNT}"
        )


    if BRONZE_UNIQUE_CODES != RECORD_COUNT:

        raise RuntimeError(
            "Bronze uniqueness reconciliation failed. "
            f"Expected={RECORD_COUNT}, "
            f"unique={BRONZE_UNIQUE_CODES}"
        )


    print("-" * 60)

    print(
        "CELL 12 COMPLETE — BRONZE MERGE SUCCESSFUL"
    )

    print("-" * 60)

    print(
        "Bronze table:",
        BRONZE_TABLE
    )

    print(
        "Rows stored:",
        BRONZE_BATCH_COUNT
    )

    print(
        "Unique PSGC codes:",
        BRONZE_UNIQUE_CODES
    )


except Exception:

    # --------------------------------------------------------
    # Record failure in ingestion metadata
    # --------------------------------------------------------

    update_run(
        metadata_table=METADATA_TABLE,
        run_id=RUN_ID,
        status="failed",
        content_checksum=CONTENT_CHECKSUM,
        raw_path=FINAL_RAW_PATH,
        record_count=RECORD_COUNT,
        page_count=PAGE_COUNT,
    )

    raise

In [0]:
# ============================================================
# CELL 13 — FINALIZE INGESTION RUN
# ============================================================

try:

    final_bronze_df = (
        spark.table(BRONZE_TABLE)
        .filter(
            F.col("content_checksum") == CONTENT_CHECKSUM
        )
    )

    FINAL_BRONZE_COUNT = final_bronze_df.count()

    FINAL_UNIQUE_CODE_COUNT = (
        final_bronze_df
        .select("psgc_code")
        .distinct()
        .count()
    )

    print("Expected records:", RECORD_COUNT)
    print("Final Bronze records:", FINAL_BRONZE_COUNT)
    print("Final unique PSGC codes:", FINAL_UNIQUE_CODE_COUNT)

    if FINAL_BRONZE_COUNT != RECORD_COUNT:
        raise RuntimeError(
            f"Cannot finalize ingestion. "
            f"Expected {RECORD_COUNT} rows, "
            f"but found {FINAL_BRONZE_COUNT}."
        )

    if FINAL_UNIQUE_CODE_COUNT != RECORD_COUNT:
        raise RuntimeError(
            f"Cannot finalize ingestion. "
            f"Expected {RECORD_COUNT} unique PSGC codes, "
            f"but found {FINAL_UNIQUE_CODE_COUNT}."
        )

    update_run(
        status="processed",
        content_checksum=CONTENT_CHECKSUM,
        raw_path=FINAL_RAW_PATH,
        record_count=RECORD_COUNT,
        page_count=PAGE_COUNT,
    )

    print("=" * 60)
    print("PSGC INGESTION COMPLETED SUCCESSFULLY")
    print("=" * 60)

    print("Run ID:", RUN_ID)
    print("Source:", SOURCE_NAME)
    print("Source version:", SOURCE_VERSION)
    print("Pages processed:", PAGE_COUNT)
    print("Records processed:", RECORD_COUNT)
    print("Content checksum:", CONTENT_CHECKSUM)
    print("Bronze table:", BRONZE_TABLE)
    print("Status: processed")

except Exception:

    update_run(
        status="failed",
        content_checksum=CONTENT_CHECKSUM,
        raw_path=FINAL_RAW_PATH,
        record_count=RECORD_COUNT,
        page_count=PAGE_COUNT,
    )

    raise

In [0]:
# ============================================================
# RECOVERY — METADATA UPDATE HELPER
# ============================================================

def sql_quote(value):
    if value is None:
        return "NULL"

    escaped = str(value).replace("'", "''")
    return f"'{escaped}'"


def update_run(
    status,
    content_checksum=None,
    raw_path=None,
    record_count=None,
    page_count=None,
):
    updates = [
        f"status = {sql_quote(status)}"
    ]

    if content_checksum is not None:
        updates.append(
            f"content_checksum = {sql_quote(content_checksum)}"
        )

    if raw_path is not None:
        updates.append(
            f"raw_file_path = {sql_quote(raw_path)}"
        )

    if record_count is not None:
        updates.append(
            f"record_count = {int(record_count)}"
        )

    if page_count is not None:
        updates.append(
            f"page_count = {int(page_count)}"
        )

    set_clause = ", ".join(updates)

    spark.sql(f"""
        UPDATE {METADATA_TABLE}
        SET {set_clause}
        WHERE batch_id = {sql_quote(RUN_ID)}
    """)


print("update_run() restored successfully.")

In [0]:
# ============================================================
# RECOVERY — DEFINE update_run()
# ============================================================

def update_run(
    status,
    content_checksum=None,
    raw_path=None,
    record_count=None,
    page_count=None,
):
    """
    Update the metadata record for the current ingestion run.
    """

    updates = [
        f"status = {sql_quote(status)}"
    ]

    if content_checksum is not None:
        updates.append(
            "content_checksum = "
            f"{sql_quote(content_checksum)}"
        )

    if raw_path is not None:
        updates.append(
            "raw_file_path = "
            f"{sql_quote(raw_path)}"
        )

    if record_count is not None:
        updates.append(
            "record_count = "
            f"{int(record_count)}"
        )

    if page_count is not None:
        updates.append(
            "page_count = "
            f"{int(page_count)}"
        )

    set_clause = ", ".join(updates)

    spark.sql(f"""
        UPDATE {METADATA_TABLE}
        SET {set_clause}
        WHERE batch_id = {sql_quote(RUN_ID)}
    """)


print("update_run() helper is ready.")

In [0]:
# ============================================================
# RECOVERY CELL — RESTORE INGESTION STATE FOR CELL 12
# ============================================================

from pyspark.sql import functions as F

# Persistent tables
METADATA_TABLE = "public_health.bronze.ingestion_metadata"
BRONZE_TABLE = "public_health.bronze.psgc"

# This is the run that successfully reached Cell 11
TARGET_RUN_ID = "6281db6c-127e-40f5-90db-47574d710430"


# ------------------------------------------------------------
# Retrieve the persisted ingestion state
# ------------------------------------------------------------

run_rows = (
    spark.table(METADATA_TABLE)
    .filter(
        F.col("batch_id") == TARGET_RUN_ID
    )
    .limit(1)
    .collect()
)


if not run_rows:
    raise RuntimeError(
        f"Ingestion run not found: {TARGET_RUN_ID}"
    )


run = run_rows[0]


# ------------------------------------------------------------
# Restore variables required by Cell 12
# ------------------------------------------------------------

RUN_ID = run["batch_id"]
SOURCE_NAME = run["source_name"]
SOURCE_VERSION = run["source_version"]
ENDPOINT = run["endpoint"]

CONTENT_CHECKSUM = run["content_checksum"]
FINAL_RAW_PATH = run["raw_file_path"]

RECORD_COUNT = int(run["record_count"])
PAGE_COUNT = int(run["page_count"])


# ------------------------------------------------------------
# Restore helper used by Cell 12 error handling
# ------------------------------------------------------------

def sql_quote(value):
    if value is None:
        return "NULL"

    escaped = str(value).replace("'", "''")
    return f"'{escaped}'"


def update_run(
    status,
    content_checksum=None,
    raw_path=None,
    record_count=None,
    page_count=None,
):
    updates = [
        f"status = {sql_quote(status)}"
    ]

    if content_checksum is not None:
        updates.append(
            f"content_checksum = {sql_quote(content_checksum)}"
        )

    if raw_path is not None:
        updates.append(
            f"raw_file_path = {sql_quote(raw_path)}"
        )

    if record_count is not None:
        updates.append(
            f"record_count = {int(record_count)}"
        )

    if page_count is not None:
        updates.append(
            f"page_count = {int(page_count)}"
        )

    set_clause = ", ".join(updates)

    spark.sql(f"""
        UPDATE {METADATA_TABLE}
        SET {set_clause}
        WHERE batch_id = {sql_quote(RUN_ID)}
    """)


# ------------------------------------------------------------
# Verify recovery
# ------------------------------------------------------------

print("Recovery successful")
print("RUN_ID:", RUN_ID)
print("Source:", SOURCE_NAME)
print("Version:", SOURCE_VERSION)
print("Bronze table:", BRONZE_TABLE)
print("Record count:", RECORD_COUNT)
print("Page count:", PAGE_COUNT)
print("Checksum:", CONTENT_CHECKSUM)
print("Final raw path:", FINAL_RAW_PATH)
print("update_run() restored:", callable(update_run))

In [0]:
%sql
SELECT
    batch_id,
    source_name,
    source_version,
    record_count,
    page_count,
    status,
    content_checksum,
    raw_file_path,
    retrieved_at
FROM public_health.bronze.ingestion_metadata
ORDER BY retrieved_at DESC;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT psgc_code) AS unique_psgc_codes
FROM public_health.bronze.psgc
WHERE content_checksum =
'6f410c890595efaf757f87c465976212216133e4b1c49f6e24ff2871881ad091';

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT psgc_code) AS unique_psgc_codes
FROM public_health.bronze.psgc;